# SMS.AC — two separately fine-tuned network software assistants

This notebook trains **two QLoRA adapters** for Qwen3-4B-Instruct-2507: request → software design, and software design text extracted from Word → explained SDLC recommendation. All model and GPU execution occurs in Kaggle.

Select **GPU T4 x2** (this recipe intentionally uses GPU 0) and enable **Internet**. Attach the private dataset containing `project.zip`. Kaggle account verification, GPU quota and Internet access must be available.

The default is a **150-step pilot per task**, not a full-dataset or production-quality claim. The supplied training pairs are weak supervision until reviewed. Completed weight files, exact data/model hashes, measured losses, paired base/adapted outputs and run tier are written to manifests. An adapter is never labeled trained just because the notebook exists.


In [ ]:
from pathlib import Path
import json, os, platform, subprocess, sys, zipfile

assert platform.system() == "Linux" and Path("/kaggle/working").is_dir(), "Run this notebook on Kaggle."
assert any(os.environ.get(k) for k in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_URL_BASE", "KAGGLE_DATA_PROXY_TOKEN")), "Kaggle environment marker is missing."

PROJECT = Path("/kaggle/working/SEproject")
RUN_FULL = False           # True trains every eligible training row for FULL_EPOCHS.
PILOT_STEPS = 150          # <=5 is recorded as smoke; >5 is pilot.
FULL_EPOCHS = 1
MAX_SEQ_LENGTH = 4096      # Complete prompt + answer; oversized rows fail, never truncate.
GRADIENT_ACCUMULATION = 8
RESUME = {"design": None, "sdlc": None}  # "auto" or checkpoint directory after an interrupted run.
TEST_LIMIT = 12            # Set 0 for all held-out examples.
GENERATION_LIMIT = 3
CHALLENGE_LIMIT = 1        # Independent prompts, not expert-validated gold answers.
MAX_NEW_TOKENS = 3072
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
BASE_REVISION = "main"     # Each run resolves this to an immutable commit and records it.

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["SMS_AC_GPU"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"  # Keep subprocess metrics visible during long runs.
os.environ["HF_HOME"] = "/tmp/sms-ac-huggingface"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_DISABLED"] = "true"
print("Run tier:", "full" if RUN_FULL else ("smoke" if PILOT_STEPS <= 5 else "pilot"))


## Load the uploaded project

The project archive may contain files directly or one `SEproject/` folder. Extraction rejects traversal, symlinks, duplicate targets and oversized archives. An existing working project is reused so checkpoint files survive a cell rerun.


In [ ]:
# Kaggle's browser uploader may expand ZIPs into an SEproject directory.
if not (PROJECT / "scripts/train_dual_qlora.py").exists():
    unpacked = [p.parent.parent for p in Path("/kaggle/input").rglob("scripts/train_dual_qlora.py")]
    if len(unpacked) > 1:
        raise RuntimeError(f"Attach exactly one project dataset: {unpacked}")
    if unpacked:
        import shutil
        shutil.copytree(unpacked[0], PROJECT, dirs_exist_ok=True)
if not (PROJECT / "scripts/train_dual_qlora.py").exists():
    accepted = {"project.zip", "seproject.zip", "sms-ac-project.zip"}
    archives = [p for p in Path("/kaggle/input").rglob("*.zip") if p.name.lower() in accepted]
    if len(archives) != 1:
        raise RuntimeError(f"Attach exactly one project.zip dataset. Candidates: {archives}")
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        entries = archive.infolist()
        if sum(item.file_size for item in entries) > 3 * 1024**3:
            raise ValueError("Project archive exceeds the 3 GiB extraction budget.")
        mapped = []
        seen = set()
        for item in entries:
            normalized = item.filename.replace("\\", "/")
            parts = Path(normalized).parts
            if parts and parts[0].lower() == "seproject":
                parts = parts[1:]
            if not parts:
                continue
            if normalized.startswith("/") or ".." in parts or any(":" in part for part in parts):
                raise ValueError(f"Unsafe archive path: {item.filename}")
            if (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError("Archive symlinks are not allowed.")
            target = PROJECT.joinpath(*parts).resolve()
            if not target.is_relative_to(PROJECT.resolve()) or target in seen:
                raise ValueError(f"Unsafe or duplicate target: {target}")
            seen.add(target)
            mapped.append((item, target))
        for item, target in mapped:
            if item.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(item) as source, target.open("wb") as destination:
                    import shutil
                    shutil.copyfileobj(source, destination)
assert (PROJECT / "requirements-kaggle.txt").exists()
for task in ("design", "sdlc"):
    for split in ("train", "validation", "test"):
        assert (PROJECT / "data/dual" / task / f"{split}.jsonl").is_file(), f"Missing {task}/{split}"
print("Project:", PROJECT)


## Install a separate, reproducible training environment

The venv and base-model cache live in `/tmp` so Kaggle outputs do not include several gigabytes of dependencies. PyTorch 2.6 with CUDA 12.4 is compatible with T4 and retains P100 support if one is assigned; the training code checks actual CUDA arithmetic and selects FP16 on T4/P100. This avoids relying on Kaggle's changing preinstalled package combinations. No FlashAttention build is required.


In [ ]:
import venv
VENV = Path("/tmp/sms-ac-training-venv")
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False).create(VENV)
# Kaggle may omit ensurepip; host pip can bootstrap an empty venv.
subprocess.run([sys.executable, "-m", "pip", "--python", str(PYTHON),
                "install", "pip==25.3"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
                "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--disable-pip-version-check",
                "-r", str(PROJECT / "requirements-kaggle.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
(PROJECT / "reports").mkdir(exist_ok=True)
lock = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze"], text=True)
(PROJECT / "reports/kaggle-environment.txt").write_text(lock)
print("Isolated training environment ready.")


## Validate complete sequences before downloading model weights

Preflight checks task labels, disjoint scenario groups, chat-template boundaries, GPU support and actual token lengths. If a complete example exceeds the budget, review the printed maximum and GPU memory before raising `MAX_SEQ_LENGTH`. Never shorten the design to a few hundred tokens; doing so destroys the requested report target.


In [ ]:
def run_script(name, *arguments):
    command = [str(PYTHON), str(PROJECT / "scripts" / name), *map(str, arguments)]
    subprocess.run(command, cwd=str(PROJECT), env=os.environ.copy(), check=True)

for task in ("design", "sdlc"):
    run_script("train_dual_qlora.py", "--task", task, "--model", BASE_MODEL,
               "--revision", BASE_REVISION, "--max-seq-length", MAX_SEQ_LENGTH,
               "--output-dir", PROJECT / "models/preflight" / task, "--preflight-only")


## Fine-tune the two specialists

Each process loads one quantized base model and saves its own adapter, tokenizer and `model_manifest.json`. Prompt tokens have zero loss weight; every assistant token is retained. Checkpoints include optimizer/trainer state and are saved every 50 steps. For a later session, attach/download prior checkpoint outputs and set `RESUME[task]` to the checkpoint directory; resume also checks the original manifest when using the same adapter output directory.

A full run may need multiple Kaggle sessions. Its manifest reports actual epochs and whether every eligible training row completed the requested epochs. GPU time/quota is never bypassed.


In [ ]:
for task in ("design", "sdlc"):
    arguments = [
        "--task", task, "--model", BASE_MODEL, "--revision", BASE_REVISION,
        "--max-seq-length", MAX_SEQ_LENGTH, "--gradient-accumulation", GRADIENT_ACCUMULATION,
        "--output-dir", PROJECT / "models" / f"{task}-adapter",
        "--save-steps", 50, "--eval-samples", 16,
    ]
    arguments += ["--full", "--epochs", FULL_EPOCHS] if RUN_FULL else ["--max-steps", PILOT_STEPS]
    if RESUME[task]:
        arguments += ["--resume-from-checkpoint", RESUME[task]]
    run_script("train_dual_qlora.py", *arguments)
    manifest = json.loads((PROJECT / "models" / f"{task}-adapter/model_manifest.json").read_text())
    assert manifest["training_completed"] is True
    print(task, manifest["run_tier"], "optimizer steps:", manifest["global_step"],
          "epochs:", manifest["trained_epochs"])


## Compare the base model with each trained adapter

Both variants receive identical held-out inputs and deterministic generation budgets. Evaluation computes assistant-token negative log likelihood and perplexity, saves Markdown outputs, records output-limit hits and includes a human review rubric. Synthetic lifecycle-label accuracy and section-keyword coverage are limited diagnostics. They do not establish expert correctness, a universally best SDLC, or improvement on real user documents.


In [ ]:
for task in ("design", "sdlc"):
    run_script("evaluate_dual_models.py", "--task", task,
               "--test-limit", TEST_LIMIT, "--generation-limit", GENERATION_LIMIT,
               "--challenge-limit", CHALLENGE_LIMIT, "--max-new-tokens", MAX_NEW_TOKENS,
               "--max-generation-context", 8192, "--rag-ablation-limit", 1)
    report_path = PROJECT / "reports/evaluation" / task / "evaluation_report.json"
    report = json.loads(report_path.read_text())
    print(task, "adapter minus base NLL:", report["adapter_minus_base_nll"],
          "incomplete generations:", report["incomplete_generations"])


## Export final adapters and evidence

The downloadable archive contains final adapters/tokenizers/manifests and measured reports. Checkpoint folders remain in the saved notebook outputs for explicit resume, but are excluded from the small final-adapter archive. Base model weights are not duplicated. Retain both adapter directories when using the application.


In [ ]:
export_path = Path("/kaggle/working/sms-ac-dual-adapters.zip")
export_files = []
for task in ("design", "sdlc"):
    adapter = PROJECT / "models" / f"{task}-adapter"
    manifest = json.loads((adapter / "model_manifest.json").read_text())
    assert manifest["training_completed"] and manifest["evaluation_status"] == "paired_heldout_complete"
    export_files.extend(p for p in adapter.iterdir() if p.is_file())
export_files.extend(p for p in (PROJECT / "reports").rglob("*") if p.is_file())
for name in ("dataset_manifest.json", "manifest.json", "DATASET_CARD.md"):
    candidate = PROJECT / "data/dual" / name
    if candidate.exists():
        export_files.append(candidate)
with zipfile.ZipFile(export_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(set(export_files)):
        archive.write(path, path.relative_to(PROJECT))
print("Saved:", export_path, "MiB:", round(export_path.stat().st_size / 1024**2, 1))
from IPython.display import FileLink, display
display(FileLink(str(export_path)))


## Official implementation references

- [Qwen model card and Apache 2.0 license](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507): 4B non-thinking model, supported by Transformers ≥4.51.
- [PEFT quantization guide](https://huggingface.co/docs/peft/developer_guides/quantization): NF4, k-bit preparation and LoRA.
- [Bitsandbytes hardware/build compatibility](https://huggingface.co/docs/bitsandbytes/installation): FP4/NF4 and compute capability requirements.
- [PyTorch previous versions](https://pytorch.org/get-started/previous-versions/): official PyTorch 2.6 CUDA 12.4 wheel index.
- [Official Kaggle kernel CLI](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md): submission, accelerators and outputs.
- [Kaggle P100 retirement announcement](https://www.kaggle.com/product-announcements/735239): T4 is the intended accelerator for this project.

A short pilot is a feasibility experiment. If the adapter underperforms the base, inspect examples and labeling quality, improve the data and rerun validation before presenting it as a better model.
